## **my_sample_02**

Drying Oven의 온도를 일정하게 유지하기 위해 PPO(Proximal Policy Optimization) 알고리즘을 사용하는 기초 프레임워크입니다. Gymnasium 라이브러리를 활용해 환경을 구축하고, Stable Baselines3로 강화학습을 구현하는 방식이 가장 효율적입니다.
여기에서는 풍압(Air Pressure)은 보통 팬 RPM과 배기구의 상태에 의해 결정되므로, 보상 함수(Reward Function)에 풍압 요소를 반영하도록 수정했습니다. 풍압이 목표 범위 내에 머물면서 온도가 일정하게 유지될 때 가장 높은 보상을 주도록 설계했습니다.


### 1. Oven 시뮬레이션 환경 정의 (Gym Custom Env)

먼저 온도, 습도, 풍압, RPM 등을 상태(State)로 갖는 가상 환경을 만들어야 합니다.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np


class DryingOvenEnv(gym.Env):
    def __init__(self):
        super(DryingOvenEnv, self).__init__()
        
        # 제어: [히터 출력 증감(-1~1), 팬 RPM 증감(-1~1)]
        self.action_space = spaces.Box(low=-1, high=1, shape=(2,), dtype=np.float32)
        
        # 상태: [내부온도, 내부습도, 배기온도, 풍압, 현재RPM]
        self.observation_space = spaces.Box(low=0, high=200, shape=(5,), dtype=np.float32)
        
        self.target_temp = 80.0     # 목표 온도
        self.target_pressure = 15.0 # 공정상 필요한 목표 풍압 (예: 15 Pa)
        self.state = np.array([25.0, 40.0, 25.0, 5.0, 1000.0])

    def step(self, action):
        heater_adj, rpm_adj = action
        curr_temp, humidity, ex_temp, air_press, rpm = self.state
        
        # 1. 물리 시뮬레이션 (간략화)
        # RPM이 변하면 풍압(Air Pressure)이 변함 (P ∝ RPM^2 관계 모사)
        new_rpm = np.clip(rpm + (rpm_adj * 100), 500, 3000)
        new_press = (new_rpm / 1000) ** 2 * 5.0  # RPM에 따른 풍압 변화
        
        # 온도는 히터에 의해 오르고, 풍압(공기 흐름)에 의해 일부 냉각됨
        new_temp = curr_temp + (heater_adj * 6.0) - (new_press * 0.2)
        
        self.state = np.array([new_temp, humidity, new_temp - 5, new_press, new_rpm])
        
        # 2. Reward 계산 (핵심 수정 부분)
        # 온도 오차 벌점 + 풍압 오차 벌점
        temp_error = abs(self.target_temp - new_temp)
        press_error = abs(self.target_pressure - new_press)
        
        # 온도를 맞추는 것이 주 목적이므로 온도에 가중치(0.8)를 더 둠
        reward = -(0.8 * temp_error + 0.2 * press_error)
        
        # 3. 종료 조건 (안전 범위를 벗어나면 종료)
        terminated = bool(new_temp < 10 or new_temp > 150 or new_press > 50)
        truncated = False
        
        return self.state, reward, terminated, truncated, {}

    def reset(self, seed=None, options=None):
        self.state = np.array([25.0, 40.0, 25.0, 5.0, 1000.0])
        return self.state, {}


### 2. 강화 학습 모델 학습 및 실행

Stable Baselines3를 사용하여 에이전트를 학습시킵니다.

In [ ]:
from stable_baselines3 import PPO

# 환경 생성
env = DryingOvenEnv()
print(f'env is defined.[{env}]')


# 모델 정의 (MlpPolicy: 다층 퍼셉트론 신경망)
model = PPO("MlpPolicy", env, verbose=1)
print(f'mode is defined. [{model}]')
# 학습 시작
model.learn(total_timesteps=5000)

print(f'model is learned. [{model}]')

test_cnt = 2

# 학습된 모델로 제어 테스트
obs, _ = env.reset()
for _ in range(10):
    obs, _ = env.reset()
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)    
    print(f"제어 액션: {action}")
    print(f"예상 상태(온도/풍압): {env.step(action)[0][0]:.2f}°C / {env.step(action)[0][3]:.2f}Pa")
    if terminated: break


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 32        |
|    ep_rew_mean     | -1.94e+03 |
| time/              |           |
|    fps             | 3137      |
|    iterations      | 1         |
|    time_elapsed    | 0         |
|    total_timesteps | 2048      |
----------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 18.4        |
|    ep_rew_mean          | -1.17e+03   |
| time/                   |             |
|    fps                  | 151         |
|    iterations           | 2           |
|    time_elapsed         | 27          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.011365855 |
|    clip_fraction        | 0.166       |
|    clip_range           | 0.2         |
|    entro

### **주요 변경 사항**

- 풍압 계산 로직: new_press가 단순히 RPM에 비례하는 것이 아니라, 실제 물리 현상처럼 RPM의 변화에 따라 연동되도록 설정했습니다.
- 보상 함수(Reward): 기존의 RPM 기반 보상을 삭제하고, target_pressure와 현재 new_press 사이의 오차를 줄이는 방향으로 학습하게 했습니다.
- 다중 목표(Multi-objective): 온도와 풍압 두 가지를 동시에 잡아야 하므로, 각각에 가중치(Weight)를 부여하여 모델이 우선순위를 판단하게 유도했습니다.
